In [0]:
%pip install flask

In [0]:
import threading
import random
from flask import Flask, jsonify, request

app = Flask(__name__)

# ── Simulated backend: 10,000 fake "orders" ────────────────────────────────
MOCK_DB = [
    {
        "id": i,
        "name": f"Order_{i}",
        "amount": round(random.uniform(10, 500), 2),
        "updated_at": "2026-09-01T10:00:00Z",
    }
    for i in range(1, 10001)
]

MAX_PAGE_SIZE = 5000  # raise/lower this to simulate different real-world API caps

@app.route("/orders")
def get_orders():
    api_key = request.headers.get("x-api-key")
    if api_key != "test-secret-123":
        return jsonify({"error": "unauthorized"}), 401

    page = int(request.args.get("page", 1))
    limit = int(request.args.get("limit", 25))
    effective_limit = min(limit, MAX_PAGE_SIZE)

    start = (page - 1) * effective_limit
    end = start + effective_limit
    batch = MOCK_DB[start:end]

    return jsonify({
        "data": batch,
        "page": page,
        "limit": effective_limit,
        "total": len(MOCK_DB),
        "has_more": end < len(MOCK_DB),
    })

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

def run_server():
    app.run(host="0.0.0.0", port=5000, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2)  # give Flask a moment to bind the port
print("Mock API server started on http://127.0.0.1:5000")

In [0]:
import requests

BASE_URL = "http://127.0.0.1:5000"

# health check
health = requests.get(f"{BASE_URL}/health")
print("Health check:", health.status_code, health.json())

# real HTTP GET with auth header, just like a real API
resp = requests.get(
    f"{BASE_URL}/orders",
    headers={"x-api-key": "test-secret-123"},
    params={"page": 1, "limit": 100},
    timeout=10,
)
print("Status:", resp.status_code)
print("Records returned:", len(resp.json()["data"]))
print("Sample record:", resp.json()["data"][0])

In [0]:
import json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
browser_host = ctx.browserHostName().get()
cluster_id = ctx.clusterId().get()
org_id = ctx.workspaceId().get()

proxy_url = f"https://{browser_host}/driver-proxy/o/{org_id}/{cluster_id}/8000/orders"
print(proxy_url)

In [0]:
import threading
import random
from flask import Flask, jsonify, request

app = Flask(__name__)

MOCK_DB = [
    {"id": i, "name": f"Order_{i}", "amount": round(random.uniform(10, 500), 2), "updated_at": "2026-09-01T10:00:00Z"}
    for i in range(1, 10001)
]
MAX_PAGE_SIZE = 5000
#describing the orders table
@app.route("/orders")
def get_orders():
    api_key = request.headers.get("x-api-key")
    if api_key != "test-secret-123":
        return jsonify({"error": "unauthorized"}), 401
    page = int(request.args.get("page", 1))
    limit = int(request.args.get("limit", 25))
    effective_limit = min(limit, MAX_PAGE_SIZE)
    start = (page - 1) * effective_limit
    end = start + effective_limit
    batch = MOCK_DB[start:end]
    return jsonify({"data": batch, "page": page, "limit": effective_limit,
                     "total": len(MOCK_DB), "has_more": end < len(MOCK_DB)})

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

#connecting the proxy to the cluster
def run_server():
    app.run(host="0.0.0.0", port=8000, use_reloader=False)   # <-- changed port

threading.Thread(target=run_server, daemon=True).start()
import time; time.sleep(2)
print("Server started on port 8000")

In [0]:
import requests

resp = requests.get(
    "http://127.0.0.1:5000/orders",   # or whichever port you set in Cell 2
    headers={"x-api-key": "test-secret-123"},
    params={"page": 1, "limit": 5000},
    timeout=10,
)
print(resp.status_code)
print(len(resp.json()["data"]), "records returned")